# Performance Estimation: the subgradient method's worst case as an SDP

This notebook turns the **adversarial perspective** from the slides into a concrete,
solvable problem. We compute the *exact* worst-case accuracy of the **averaged
subgradient method** over nonsmooth convex $L$-Lipschitz functions, and check it against
the Lecture-8 bound
$$ f(x_A) - f(x_\star) \;\le\; E(h) \;=\; \frac{R^2 + L^2\sum_{i=0}^N h_i^2}{2\sum_{i=0}^N h_i}. $$

Throughout we fix $R = L = 1$ and $N = 10$.

*References.* Y. Drori, M. Teboulle, *Performance of first-order methods for smooth convex
minimization: a novel approach*, Math. Prog. 2014. &nbsp; A. Taylor, J. Hendrickx, F. Glineur,
*Smooth strongly convex interpolation and exact worst-case performance of first-order methods*,
Math. Prog. 2017.

## Setup (as on the slides)

Subgradient descent queries $g_i \in \partial f(x_i)$ and updates
$$ x_{i+1} = x_i - h_i g_i, \qquad x_A = \frac{\sum_{i=0}^N h_i\,x_i}{\sum_{i=0}^N h_i}. $$
The proof sees only the sampled data $f_i = f(x_i)$ and $g_i$, for $i \in \{0,\dots,N,A,\star\}$.
The optimal constant stepsize is $h_i = R/(L\sqrt{N+1})$, for which $E(h) = RL/\sqrt{N+1}$.

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt

R, L, N = 1.0, 1.0, 10
h   = np.full(N + 1, R / (L * np.sqrt(N + 1)))   # constant optimal stepsize h_0..h_N
lam = h / h.sum()                                # averaging weights lambda_i = h_i / sum_j h_j

E_h = (R**2 + L**2 * np.sum(h**2)) / (2 * np.sum(h))
print(f"E(h)         = {E_h:.6f}")
print(f"RL/sqrt(N+1) = {R * L / np.sqrt(N + 1):.6f}")

## The lifted variables: a Gram matrix $G$ and values $F$

Stack the vectors of interest as the columns of $P$ and let $G = P^\top P \succeq 0$ collect
their pairwise inner products,
$$ P = \big[\,x_0, \dots, x_N, x_{N+1},\, x_\star,\ g_0, \dots, g_N,\, g_A\,\big],
   \qquad G_{kl} = \langle p_k, p_l\rangle . $$

Two points matching the slide discussion:

* **$x_A$ is *not* a separate column.** The averaging identity $x_A = \sum_i \lambda_i x_i$ makes
  it an affine combination of columns already present, so we substitute it directly. (Forcing
  $\lVert x_A - \sum_i \lambda_i x_i\rVert^2 = 0$ instead would make $G$ singular and break the
  solver's Slater condition.)
* **$x_{N+1}$ *is* a column.** The telescoping uses the step at $i = N$, i.e.
  $x_{N+1} = x_N - h_N g_N$; the final iterate is discarded by the algorithm but the analysis sees it.

Every vector is a coordinate vector in this basis, and any inner product
$\langle a, b\rangle = a^\top G\,b$ is **linear in $G$**.

In [ ]:
# --- column layout of P ---
idx, col = {}, 0
for i in range(N + 2):            # x_0, ..., x_N, x_{N+1}
    idx[("x", i)] = col; col += 1
idx[("x", "*")] = col; col += 1   # x_star
for i in range(N + 1):            # g_0, ..., g_N
    idx[("g", i)] = col; col += 1
idx[("g", "A")] = col; col += 1   # g_A
m = col                           # number of columns of P

def vec(key):
    "coordinate vector e_key in the P-basis"
    v = np.zeros(m); v[idx[key]] = 1.0
    return v

x  = [vec(("x", i)) for i in range(N + 2)]    # x[0], ..., x[N+1]
xs = vec(("x", "*"))                           # x_star
g  = [vec(("g", i)) for i in range(N + 1)]     # g[0], ..., g[N]
gA = vec(("g", "A"))                           # g_A
xA = sum(lam[i] * x[i] for i in range(N + 1))  # x_A = sum_i lambda_i x_i  (eliminated)

# --- lifted decision variables ---
G = cp.Variable((m, m), symmetric=True)        # Gram matrix, G = P^T P
F = cp.Variable(N + 3)                          # F = (f_0, ..., f_N, f_A, f_star)
f, fA, fs = [F[i] for i in range(N + 1)], F[N + 1], F[N + 2]

def ip(a, b):
    "inner product <a, b> = a^T G b   (linear in G)"
    return a @ G @ b

## Constraints (exactly the adversarial-perspective slide)

Colour-coded as on the slide:

* <span style="color:#1f77b4">**blue**</span> &mdash; the descent recursion
  $\lVert x_{i+1}-x_\star\rVert^2 \le \lVert x_i-x_\star\rVert^2 - 2h_i\langle g_i, x_i-x_\star\rangle + h_i^2\lVert g_i\rVert^2$;
* <span style="color:#2ca02c">**green**</span> &mdash; the start $\lVert x_0-x_\star\rVert^2 \le R^2$;
* <span style="color:#d62728">**red**</span> &mdash; the two families of subgradient inequalities and the
  Lipschitz bound $\lVert g_i\rVert^2 \le L^2$;
* and $G \succeq 0$, the only constraint tying $G$ back to genuine vectors.

In [ ]:
cons = [G >> 0]

# blue: descent recursion, i = 0..N  (the i = N step uses x_{N+1})
for i in range(N + 1):
    di, di1 = x[i] - xs, x[i + 1] - xs
    cons += [ip(di1, di1) <= ip(di, di) - 2 * h[i] * ip(g[i], di) + h[i]**2 * ip(g[i], g[i])]

# green: starting condition
cons += [ip(x[0] - xs, x[0] - xs) <= R**2]

# red: subgradient inequality between x_i and x_star
for i in range(N + 1):
    cons += [ip(g[i], xs - x[i]) <= fs - f[i]]

# red: subgradient inequality between x_A and x_i   (sums to f_A <= sum_i lambda_i f_i)
for i in range(N + 1):
    cons += [ip(gA, x[i] - xA) <= f[i] - fA]

# red: Lipschitz bound
for i in range(N + 1):
    cons += [ip(g[i], g[i]) <= L**2]

print(f"{len(cons)} constraints; G is {m} x {m}")

In [ ]:
prob = cp.Problem(cp.Maximize(fA - fs), cons)
pep_value = prob.solve(solver=cp.CLARABEL)

print(f"status         = {prob.status}")
print(f"PEP worst case = {pep_value:.6f}")
print(f"E(h)           = {E_h:.6f}")
print(f"difference     = {abs(pep_value - E_h):.2e}")

The worst case equals $E(h)$ to solver tolerance: **the Lecture-8 bound is tight**. There is a
nonsmooth convex $L$-Lipschitz function on which averaged subgradient descent is exactly this far
from optimal.

We can even *recover* it: any $G \succeq 0$ factors as $G = P^\top P$ (eigendecomposition), and the
columns of $P$ are explicit worst-case iterates and subgradients.

In [ ]:
Gv = G.value
w, V = np.linalg.eigh(Gv)
w = np.clip(w, 0.0, None)
Pmat = (V * np.sqrt(w)).T                       # column j is the vector p_j in R^m
rank = int((w > 1e-7 * w.max()).sum())

xv  = [Pmat[:, idx[("x", i)]] for i in range(N + 2)]
xsv =  Pmat[:, idx[("x", "*")]]
gv  = [Pmat[:, idx[("g", i)]] for i in range(N + 1)]

print(f"worst-case dimension  d = rank(G) = {rank}")
print(f"||x_0 - x_*||         = {np.linalg.norm(xv[0] - xsv):.4f}   (start,     <= R = {R})")
print(f"max_i ||g_i||         = {max(np.linalg.norm(gi) for gi in gv):.4f}   (Lipschitz, <= L = {L})")

The worst case lives in a **high-dimensional** space ($d = \mathrm{rank}\,G$ grows with $N$) &mdash;
exactly the freedom we granted the adversary by never fixing where the iterates live.

## Tightness across $N$

Wrapping the build in a function and re-solving over a range of $N$ shows the SDP value coincides
with $RL/\sqrt{N+1}$ throughout.

In [ ]:
def pep_subgradient(N, h, R=1.0, L=1.0, solver=cp.CLARABEL):
    "Worst-case f(x_A) - f(x_*) of the averaged subgradient method, as an SDP."
    lam = h / h.sum()
    idx, col = {}, 0
    for i in range(N + 2): idx[("x", i)] = col; col += 1
    idx[("x", "*")] = col; col += 1
    for i in range(N + 1): idx[("g", i)] = col; col += 1
    idx[("g", "A")] = col; col += 1
    m = col
    def vec(k):
        v = np.zeros(m); v[idx[k]] = 1.0; return v
    x  = [vec(("x", i)) for i in range(N + 2)]; xs = vec(("x", "*"))
    g  = [vec(("g", i)) for i in range(N + 1)]; gA = vec(("g", "A"))
    xA = sum(lam[i] * x[i] for i in range(N + 1))
    G = cp.Variable((m, m), symmetric=True); F = cp.Variable(N + 3)
    f, fA, fs = [F[i] for i in range(N + 1)], F[N + 1], F[N + 2]
    ip = lambda a, b: a @ G @ b
    cons = [G >> 0, ip(x[0] - xs, x[0] - xs) <= R**2]
    for i in range(N + 1):
        di, di1 = x[i] - xs, x[i + 1] - xs
        cons += [ip(di1, di1) <= ip(di, di) - 2 * h[i] * ip(g[i], di) + h[i]**2 * ip(g[i], g[i])]
        cons += [ip(g[i], xs - x[i]) <= fs - f[i]]
        cons += [ip(gA, x[i] - xA) <= f[i] - fA]
        cons += [ip(g[i], g[i]) <= L**2]
    return cp.Problem(cp.Maximize(fA - fs), cons).solve(solver=solver)

Ns  = np.arange(1, 26)
pep = np.array([pep_subgradient(n, np.full(n + 1, 1.0 / np.sqrt(n + 1))) for n in Ns])

plt.figure(figsize=(6, 4))
plt.plot(Ns, pep, "o", ms=6, label="PEP (SDP worst case)")
plt.plot(Ns, 1.0 / np.sqrt(Ns + 1), "-", label=r"$RL/\sqrt{N+1}$")
plt.xlabel("$N$"); plt.ylabel(r"$f(x_A) - f(x_\star)$")
plt.title("Averaged subgradient method: worst case vs. textbook bound")
plt.legend(); plt.tight_layout(); plt.show()

print("max |PEP - RL/sqrt(N+1)| =", np.max(np.abs(pep - 1.0 / np.sqrt(Ns + 1))))